# Deep Agents + Elasticsearch: Long-Running Research Pipelines

This notebook demonstrates how to use LangChain's **Deep Agents** framework
to run multi-step research queries and persist findings in **Elasticsearch**.

**Use case:** A research agent that investigates a scientific question
(e.g., best LLM for biochemical analysis), stores sources and findings
in Elasticsearch, and produces a structured report.

## When to Use LangGraph, LangChain, or Deep Agents

| Framework | Best For | Example |
|-----------|----------|---------|
| **LangChain** | Simple chains: prompt → LLM → output | Summarize a document, answer a question |
| **LangGraph** | Custom stateful workflows with branching and cycles | Multi-step approval pipelines, chatbots with tool use |
| **Deep Agents** | Long-running, autonomous research that needs planning, sub-agents, and memory | "Research the top 5 LLMs for biomedical NLP and write a report" |

Deep Agents build **on top of** LangGraph. They add middleware for planning
(to-do lists), file I/O, and sub-agent delegation, things you'd otherwise
wire manually in LangGraph.

In this notebook we use Deep Agents because our task requires the agent to
**plan its own research steps**, search the web iteratively, and accumulate
findings over time, exactly the pattern Deep Agents automate.

## Setup

In [19]:
%pip install deepagents langchain-elasticsearch langchain-google-genai python-dotenv tavily-python -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
from dotenv import load_dotenv

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

## Initialize Elasticsearch Client

In [21]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)

es.info()

ObjectApiResponse({'name': 'instance-0000000004', 'cluster_name': 'fd31ad5f77d841d69b622c17ed64b766', 'cluster_uuid': 'YxmDkf9DSwWpvcCLv98q8A', 'version': {'number': '9.3.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '17b451d8979a29e31935fe1eb901310350b30e62', 'build_date': '2026-01-29T10:05:46.708397977Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

## Define the Research Index

We use Elastic Inference Service's built-in Jina Embeddings v5 endpoint (`.jina-embeddings-v5-text-small`) for semantic search. Text fields use `copy_to` to populate the `semantic_field`.

In [15]:
INDEX_NAME = "deep-agent-research"
INFERENCE_ID = ".jina-embeddings-v5-text-small"

index_body = {
    "mappings": {
        "properties": {
            "query": {"type": "text", "copy_to": "semantic_field"},
            "source": {"type": "keyword"},
            "title": {
                "type": "text",
                "fields": {"keyword": {"type": "keyword"}},
                "copy_to": "semantic_field",
            },
            "content": {"type": "text", "copy_to": "semantic_field"},
            "timestamp": {"type": "date"},
            "tags": {"type": "keyword"},
            "semantic_field": {
                "type": "semantic_text",
                "inference_id": INFERENCE_ID,
            },
        }
    }
}

if not es.indices.exists(index=INDEX_NAME):
    es.indices.create(index=INDEX_NAME, body=index_body)
    print(f"Created index: {INDEX_NAME}")
else:
    print(f"Index already exists: {INDEX_NAME}")

Index already exists: deep-agent-research


## Build Custom Tools for the Agent

The agent needs two capabilities:
1. **Search the web**: via Tavily ([LangChain partner](https://docs.tavily.com/documentation/integrations/langchain) for web search )
2. **Store findings**: save results to Elasticsearch

In [22]:
from datetime import datetime, timezone

from langchain_core.tools import tool
from tavily import TavilyClient

tavily = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def web_search(query: str) -> str:
    """Search the web for information on a topic."""

    results = tavily.search(query=query, max_results=5)
    formatted = []

    for r in results["results"]:
        formatted.append(f"**{r['title']}**\nURL: {r['url']}\n{r['content'][:500]}")

    return "\n\n---\n\n".join(formatted)


@tool
def store_finding(
    title: str, source: str, content: str, tags: list[str], query: str
) -> str:
    """Store a research finding in Elasticsearch."""

    doc = {
        "query": query,
        "source": source,
        "title": title,
        "content": content,
        "tags": tags,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    resp = es.index(index=INDEX_NAME, document=doc)

    return f"Stored finding '{title}' (id: {resp['_id']})"

## Create the Deep Agent

> **Note:** Preview models expire periodically. Check the [available models list](https://ai.google.dev/gemini-api/docs/models) and use a recent one.

In [40]:
from deepagents import create_deep_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

agent = create_deep_agent(
    model=model,
    tools=[web_search, store_finding],
    system_prompt=(
        "You are a research assistant with access to two tools: "
        "web_search and store_finding.\n\n"
        "For EVERY query you MUST:\n"
        "1. Use web_search to find relevant sources.\n"
        "2. For EACH useful result, call store_finding to save it to "
        "Elasticsearch.\n"
        "3. After storing all findings, write a brief summary.\n\n"
        "Always pass the original user query in the 'query' parameter."
    ),
)

## Run Research Queries

In [ ]:
research_query = (
    "What is the best LLM for biochemical analysis of cancer cells in mice?"
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": research_query}]},
)

print(result["messages"][-1].content)

[{'type': 'text', 'text': 'The "best" LLM for biochemical analysis of cancer cells in mice depends on the specific type of data being analyzed (e.g., transcriptomics, proteomics, or metabolomics). While general-purpose models like **GPT-4o** or **Med-PaLM 2** provide strong reasoning capabilities, several specialized LLM-driven frameworks have been developed for specific biochemical tasks:\n\n### 1. **CellAgent** (Best for Single-Cell & Spatial Analysis)\n**CellAgent** is an LLM-driven multi-agent framework specifically designed for automated **single-cell RNA sequencing (scRNA-seq)** and **spatial transcriptomics**. It uses LLMs to autonomously select the best tools and hyperparameters for analyzing cell heterogeneity, which is a core part of studying cancer cell behavior in mouse models.\n*   **Key Strength:** Automates end-to-end analysis of single-cell data through natural language.\n*   **Relevance:** Highly effective for identifying cell types and states in murine tumor microenvi

## Explore Stored Research

Let's review what the agent stored, then use hybrid search and aggregations.

In [42]:
es.indices.refresh(index=INDEX_NAME)

all_docs = es.search(
    index=INDEX_NAME,
    body={"query": {"match_all": {}}, "sort": [{"timestamp": "asc"}], "size": 50},
)

print(f"Total findings stored: {all_docs['hits']['total']['value']}\n")
for hit in all_docs["hits"]["hits"]:
    doc = hit["_source"]
    print(f"[{doc['timestamp'][:19]}] {doc['title']}")
    print(f"  Source: {doc['source']}")
    print(f"  Query:  {doc['query'][:80]}")
    print()

Total findings stored: 5

[2026-03-18T16:48:28] CellAgent: LLM-Driven Multi-Agent Framework for Single-Cell Analysis
  Source: arXiv:2407.09811, bioRxiv 2024.05.13.593861v4
  Query:  best LLM for biochemical analysis of cancer cells in mice

[2026-03-18T16:48:28] PROTEUS: LLM-Based System for Automating Proteomics Research
  Source: arXiv:2411.03743, PMC12806030
  Query:  best LLM for biochemical analysis of cancer cells in mice

[2026-03-18T16:48:28] CLAW-MRM: LLM-Aided Lipidomics Profiling
  Source: ACS Analytical Chemistry 2024 (10.1021/acs.analchem.4c05039)
  Query:  best LLM for biochemical analysis of cancer cells in mice

[2026-03-18T16:48:28] DrBioRight 2.0: LLM-Powered Bioinformatics Chatbot for Cancer Omics
  Source: Nature Communications 2025 (s41467-025-57430-4)
  Query:  best LLM for biochemical analysis of cancer cells in mice

[2026-03-18T16:48:28] Med-PaLM 2 and Med-PaLM M: State-of-the-Art Medical LLMs
  Source: Nature 2023, ResearchGate 2024
  Query:  best LLM for bio

### Hybrid Search with RRF

Combine BM25 (lexical) and semantic search using Reciprocal Rank Fusion.

In [43]:
es.indices.refresh(index=INDEX_NAME)

search_text = "biochemical analysis cancer cells mice"

response = es.search(
    index=INDEX_NAME,
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "standard": {
                            "query": {
                                "multi_match": {
                                    "query": search_text,
                                    "fields": ["query", "title", "content"],
                                }
                            }
                        }
                    },
                    {
                        "standard": {
                            "query": {
                                "semantic": {
                                    "field": "semantic_field",
                                    "query": search_text,
                                }
                            }
                        }
                    },
                ],
                "rank_window_size": 50,
                "rank_constant": 20,
            }
        },
        "size": 10,
    },
)

print(f"Found {response['hits']['total']['value']} findings (hybrid RRF):\n")
for hit in response["hits"]["hits"]:
    doc = hit["_source"]
    print(f"  - [{doc['source']}] {doc['title']}")
    print(f"    Tags: {', '.join(doc.get('tags', []))}")
    print()

Found 5 findings (hybrid RRF):

  - [arXiv:2411.03743, PMC12806030] PROTEUS: LLM-Based System for Automating Proteomics Research
    Tags: PROTEUS, proteomics, biochemical analysis, cancer research

  - [ACS Analytical Chemistry 2024 (10.1021/acs.analchem.4c05039)] CLAW-MRM: LLM-Aided Lipidomics Profiling
    Tags: CLAW-MRM, lipidomics, metabolomics, biochemical analysis, cancer research

  - [Nature 2023, ResearchGate 2024] Med-PaLM 2 and Med-PaLM M: State-of-the-Art Medical LLMs
    Tags: Med-PaLM 2, Med-PaLM M, medical LLM, cancer research

  - [arXiv:2407.09811, bioRxiv 2024.05.13.593861v4] CellAgent: LLM-Driven Multi-Agent Framework for Single-Cell Analysis
    Tags: CellAgent, single-cell analysis, transcriptomics, cancer research, mice

  - [Nature Communications 2025 (s41467-025-57430-4)] DrBioRight 2.0: LLM-Powered Bioinformatics Chatbot for Cancer Omics
    Tags: DrBioRight, bioinformatics, proteomics, cancer omics



## Aggregate by Tags

In [44]:
agg_response = es.search(
    index=INDEX_NAME,
    body={
        "size": 0,
        "aggs": {"top_tags": {"terms": {"field": "tags", "size": 10}}},
    },
)

print("Top tags across all findings:")
for bucket in agg_response["aggregations"]["top_tags"]["buckets"]:
    print(f"  {bucket['key']}: {bucket['doc_count']}")

Top tags across all findings:
  cancer research: 4
  biochemical analysis: 2
  proteomics: 2
  CLAW-MRM: 1
  CellAgent: 1
  DrBioRight: 1
  Med-PaLM 2: 1
  Med-PaLM M: 1
  PROTEUS: 1
  bioinformatics: 1


In [ ]:
second_query = "What is the best front-end framework to connect to a DeepAgent service for assigning tasks and generating research documents?"

result2 = agent.invoke(
    {"messages": [{"role": "user", "content": second_query}]},
)

print(result2["messages"][-1].content)

## Cleanup

In [ ]:
# Delete the research index
es.options(ignore_status=[400, 404]).indices.delete(index=INDEX_NAME)
print(f"Deleted index: {INDEX_NAME}")

# Close the Elasticsearch connection
es.close()
print("Elasticsearch connection closed.")